# Week 3 hand-in — network loaders (optional)

**27200 Data-driven Bioengineering**

The group assignment can be done entirely in Cytoscape. This notebook is for groups that would
rather work in Python. It gets each of the six networks into a `networkx` graph, in one of two ways:

- **Option A — from `networks.zip`** (the file on DTU Learn). Same data as everyone else, no
  server involved. Recommended.
- **Option B — live from the public sources.** One function per network, in case you want to change
  a threshold or see where the data comes from.

Run the setup cell, then **only the cells for your network**. Every loader returns a plain
`networkx` graph, so everything from Exercise 2 (`degree`, `clustering`, `shortest_path_length`,
the degree-distribution plot, the FFL counter and the randomiser) works on it unchanged. Copy those
cells across.

> **Terms**
> **Directed / undirected** — whether an edge has a direction (TF → gene) or not (A binds B).
> **Signed** — each edge is marked activating or inhibiting.
> **Weighted** — each edge carries a number (here: how many synapses connect two neurons).
> **Bipartite** — two kinds of node, and edges only between kinds (metabolite — reaction). You must
> decide how to turn that into a one-kind graph before most measures make sense.


In [1]:
!pip install -q networkx

zsh:1: command not found: pip


---
## Option A — load from `networks.zip`

Upload `networks.zip` from DTU Learn when the file chooser appears (in Colab: the cell below opens
it; elsewhere, put the zip next to this notebook). The cell unzips it and lists the folders.

Then `load_sif()` turns any `.sif` in the zip into a graph. A SIF line is `node1  type  node2`; the
type becomes the edge attribute `kind` (activates / represses, chemical / electrical, …). Networks
1–4 are directed, 5 and 6 undirected — the function picks that from the folder name, but you can
override it.


In [ ]:
import os, zipfile, glob
import numpy as np, pandas as pd, networkx as nx

if not os.path.exists("networks"):
    if not os.path.exists("networks.zip"):
        try:
            from google.colab import files
            files.upload()                       # pick networks.zip on your computer
        except ImportError:
            raise FileNotFoundError("Put networks.zip next to this notebook first.")
    zipfile.ZipFile("networks.zip").extractall(".")
print("\n".join(sorted(glob.glob("networks/*/"))))

def load_sif(path, directed=None):
    '''Read a Cytoscape .sif into networkx. The edge type becomes the attribute "kind".'''
    if directed is None:                          # folders 1-4 are directed, 5-6 undirected
        directed = os.path.basename(os.path.dirname(path))[0] in "1234"
    G = nx.DiGraph() if directed else nx.Graph()
    with open(path) as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) == 1: G.add_node(parts[0])          # isolated node
            else: G.add_edge(parts[0], parts[2], kind=parts[1])
    return G

def load_edges_tsv(path, directed=True):
    '''Read a *_edges.tsv: first two columns are source and target, the rest become edge attributes.'''
    df = pd.read_csv(path, sep="\t")
    src, tgt = df.columns[:2]
    return nx.from_pandas_edgelist(df, src, tgt, edge_attr=True,
                                   create_using=nx.DiGraph() if directed else nx.Graph())

def describe(G, name):
    '''One-paragraph summary of any graph, directed or not.'''
    n, m = G.number_of_nodes(), G.number_of_edges()
    U = G.to_undirected() if G.is_directed() else G
    comps = sorted(nx.connected_components(U), key=len, reverse=True)
    deg = np.array([d for _, d in U.degree()])
    print(f"{name}\n  {n} nodes, {m} edges, {'directed' if G.is_directed() else 'undirected'}\n"
          f"  degree: mean {deg.mean():.1f}, max {deg.max()}\n"
          f"  {len(comps)} connected component(s); largest has {len(comps[0])} nodes "
          f"({100*len(comps[0])/n:.0f}%)\n  average clustering {nx.average_clustering(U):.3f}")
    return G


In [ ]:
# --- pick your network: change the path ---------------------------------------
G = describe(load_sif("networks/1_yeast_trn/yeast_trn.sif"), "Yeast transcription network")
R = describe(load_sif("networks/1_yeast_trn/yeast_trn_RANDOM.sif"), "same size, random")

# other examples:
#   G = load_sif("networks/2_human_trn/human_trn.sif")
#   G = load_edges_tsv("networks/4_celegans_connectome/celegans_connectome_edges.tsv")   # keeps n_synapses
#       (a chemical and an electrical link between the same pair collapse into one edge — filter the
#        "type" column of the TSV first if you want only one kind)
#   G = load_sif("networks/6_ecoli_core/ecoli_core_metabolite_graph_NO_CURRENCY.sif")
# node tables (worm cell classes; E. coli names and currency flag):
#   nodes = pd.read_csv("networks/4_celegans_connectome/celegans_nodes.tsv", sep="\t")


---
## Option B — fetch live from the public sources

One function per network. Each downloads the data, builds the graph and prints a description.
These are the steps that produced the files in `networks.zip` (retrieved 13 September 2026); run
one only if you want to change something — a confidence threshold, an edge type — or to see where
the data comes from. Servers occasionally time out; if one does, use Option A.


In [2]:
import io, json, requests
import numpy as np, pandas as pd, networkx as nx
H = {"User-Agent": "27200-course"}

def get(url, **kw):
    '''Download text from a URL (with optional query parameters).'''
    r = requests.get(url, timeout=120, headers=H, **kw); r.raise_for_status(); return r.text

def describe(G, name):
    '''One-paragraph summary of any graph, directed or not.'''
    n, m = G.number_of_nodes(), G.number_of_edges()
    kind = "directed" if G.is_directed() else "undirected"
    U = G.to_undirected() if G.is_directed() else G
    comps = sorted(nx.connected_components(U), key=len, reverse=True)
    deg = np.array([d for _, d in U.degree()])
    print(f"{name}\n  {n} nodes, {m} edges, {kind}")
    print(f"  mean degree {deg.mean():.2f}, max degree {deg.max()}, average clustering {nx.average_clustering(U):.3f}")
    print(f"  {len(comps)} connected component(s); largest has {len(comps[0])} nodes ({100*len(comps[0])/n:.0f}%)")
    return G

def to_sif(G, path, kind="pp"):
    '''Write a SIF file for Cytoscape. Edge attribute "kind" is used if present.'''
    with open(path, "w") as f:
        for a, b, d in G.edges(data=True):
            f.write(f"{a}\t{d.get('kind', kind)}\t{b}\n")
    print("wrote", path)

print("ready")

ready


---
## Network 1 — Yeast transcription regulatory network

*Saccharomyces cerevisiae*, transcription factor → target gene. The dataset behind Milo et al.
2002 — the one you counted feed-forward loops in during Exercise 2. Directed, unsigned. Node labels
are numeric IDs from the original file.

Source: [Alon lab, collection of complex networks](https://www.weizmann.ac.il/mcb/UriAlon/download/collection-complex-networks).

In [3]:
def load_yeast_trn():
    url = ("https://www.weizmann.ac.il/mcb/UriAlon/sites/mcb.UriAlon/files/uploads/"
           "CollectionsOfComplexNetwroks/yeastinter_st.txt")
    edges = [ln.split()[:2] for ln in get(url).strip().split("\n")]   # columns: source target (weight)
    G = nx.DiGraph(edges)
    G.remove_edges_from(nx.selfloop_edges(G))                          # autoregulation, set aside
    return describe(G, "Yeast transcription network (Alon lab, 2002)")

G = load_yeast_trn()

Yeast transcription network (Alon lab, 2002)
  688 nodes, 1079 edges, directed
  mean degree 3.13, max degree 71, average clustering 0.047
  11 connected component(s); largest has 662 nodes (96%)


---
## Network 2 — Human transcription regulatory network

Human transcription factor → target gene, from **DoRothEA** (Garcia-Alonso et al. 2019), confidence
levels A and B only — the curated, literature-supported part. Directed and **signed**: each edge
says whether the TF activates or represses the target. Node labels are gene symbols.

Source: [OmniPath](https://omnipathdb.org), `datasets=dorothea`.

In [4]:
def load_human_trn(levels="A,B"):
    txt = get("https://omnipathdb.org/interactions",
              params={"datasets": "dorothea", "dorothea_levels": levels, "genesymbols": "1",
                      "organisms": "9606", "fields": "dorothea_level"})
    df = pd.read_csv(io.StringIO(txt), sep="\t")
    G = nx.DiGraph()
    for r in df.itertuples():
        sign = "activates" if r.is_stimulation == 1 else ("represses" if r.is_inhibition == 1 else "unknown")
        G.add_edge(r.source_genesymbol, r.target_genesymbol, kind=sign, level=r.dorothea_level)
    G.remove_edges_from(nx.selfloop_edges(G))
    return describe(G, f"Human TF-target network (DoRothEA levels {levels})")

G = load_human_trn()
tfs = [n for n, d in G.out_degree() if d > 0]
print(f"  {len(tfs)} transcription factors; edge signs:", pd.Series([d['kind'] for _,_,d in G.edges(data=True)]).value_counts().to_dict())

Human TF-target network (DoRothEA levels A,B)
  5252 nodes, 15117 edges, directed
  mean degree 5.73, max degree 723, average clustering 0.168
  9 connected component(s); largest has 5231 nodes (100%)
  367 transcription factors; edge signs: {'unknown': 8256, 'activates': 6086, 'represses': 775}


---
## Network 3 — Human signalling network

Protein → protein signalling interactions: phosphorylation, activation, inhibition, complex
formation — the "activation cascade" layer. From **OmniPath**'s curated core. Directed and signed.
It is large (~7,000 proteins, ~70,000 edges), so for anything that scales badly — motif counting,
randomisation, betweenness — work on a **subnetwork**: a pathway's proteins, or a seed protein and its
neighbours within two steps. `nx.ego_graph(G, "EGFR", radius=2)` is a good start.

Source: [OmniPath](https://omnipathdb.org), `datasets=omnipath`.

In [5]:
def load_human_signalling():
    txt = get("https://omnipathdb.org/interactions",
              params={"datasets": "omnipath", "genesymbols": "1", "organisms": "9606",
                      "directed": "1", "signed": "1", "fields": "sources,references"})
    df = pd.read_csv(io.StringIO(txt), sep="\t")
    G = nx.DiGraph()
    for r in df.itertuples():
        sign = "stimulates" if r.is_stimulation == 1 else ("inhibits" if r.is_inhibition == 1 else "unknown")
        G.add_edge(r.source_genesymbol, r.target_genesymbol, kind=sign)
    G.remove_edges_from(nx.selfloop_edges(G))
    return describe(G, "Human signalling network (OmniPath)")

G = load_human_signalling()
ego = nx.ego_graph(G, "EGFR", radius=2)          # example subnetwork: EGFR and everything within two steps
print(f"  example: EGFR within 2 steps -> {ego.number_of_nodes()} proteins, {ego.number_of_edges()} edges")

Human signalling network (OmniPath)
  7154 nodes, 71789 edges, directed


  mean degree 19.77, max degree 356, average clustering 0.082
  80 connected component(s); largest has 6945 nodes (97%)
  example: EGFR within 2 steps -> 770 proteins, 4430 edges


---
## Network 4 — *C. elegans* neuronal connectome

Every neuron of the hermaphrodite worm, and every synapse between them — the only complete
connectome of an animal, and the one Alon uses in chapter 4 to show that the same motifs recur in
neurons and in transcription. Directed and **weighted** (number of synapses), with two edge types:
**chemical** synapses (directed) and **electrical** gap junctions (symmetric). The file also
includes the muscles neurons connect to; decide whether to keep them.

Source: [OpenWorm ConnectomeToolbox](https://github.com/openworm/ConnectomeToolbox), from
WormAtlas / Cook et al. 2019.

In [6]:
def load_celegans(edge_type="chemical"):
    '''edge_type: "chemical", "electrical", or "both".'''
    url = "https://raw.githubusercontent.com/openworm/ConnectomeToolbox/main/cect/data/herm_full_edgelist.csv"
    df = pd.read_csv(io.StringIO(get(url)))
    df.columns = [c.strip() for c in df.columns]
    for c in ["Source", "Target", "Type"]: df[c] = df[c].str.strip()
    if edge_type != "both": df = df[df.Type == edge_type]
    G = nx.DiGraph()
    for r in df.itertuples():
        G.add_edge(r.Source, r.Target, weight=float(r.Weight), kind=r.Type)
        if r.Type == "electrical": G.add_edge(r.Target, r.Source, weight=float(r.Weight), kind=r.Type)   # gap junctions go both ways
    return describe(G, f"C. elegans connectome ({edge_type} synapses)")

G = load_celegans("chemical")

C. elegans connectome (chemical synapses)
  419 nodes, 4681 edges, directed
  mean degree 19.14, max degree 91, average clustering 0.315
  2 connected component(s); largest has 380 nodes (91%)


---
## Network 5 — Human kinase interactome

Physical protein–protein interactions among the human protein kinases (the "kinome"), from
**STRING** at high confidence (score ≥ 700). Undirected. Two live queries: UniProt supplies the
list of reviewed human kinases, STRING supplies the interactions among them. The kinome is a useful
choice because you met two of its members — CDK2 and GSK3B — in the Week 2 assignment, and because
kinases sit at the centre of signalling.

Sources: [UniProt](https://www.uniprot.org) (keyword *Kinase*), [STRING](https://string-db.org).

In [7]:
def load_kinome_ppi(score=700):
    u = ("https://rest.uniprot.org/uniprotkb/search?query=(keyword:KW-0418)%20AND%20(organism_id:9606)"
         "%20AND%20(reviewed:true)&fields=gene_primary&format=tsv&size=500")
    genes = [l.split("\t")[0] for l in get(u).strip().split("\n")[1:] if l.strip()]
    r = requests.post("https://string-db.org/api/tsv/network", timeout=120,
                      data={"identifiers": "\r".join(genes), "species": 9606, "required_score": score,
                            "network_type": "physical", "caller_identity": "27200_course"})
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text), sep="\t")
    G = nx.Graph()
    for r_ in df.itertuples(): G.add_edge(r_.preferredName_A, r_.preferredName_B, score=r_.score)
    return describe(G, f"Human kinome physical PPI (STRING, score >= {score})")

G = load_kinome_ppi()

Human kinome physical PPI (STRING, score >= 700)
  272 nodes, 470 edges, undirected
  mean degree 3.46, max degree 27, average clustering 0.404
  35 connected component(s); largest has 184 nodes (68%)


---
## Network 6 — *E. coli* core metabolism

The textbook *E. coli* core model: 72 metabolites, 95 reactions — glycolysis, TCA, pentose
phosphate, oxidative phosphorylation. It is **bipartite**: metabolites connect to reactions, never
directly to each other. How you turn that into a graph is *the* decision in this assignment:

- **metabolite graph** — two metabolites joined if some reaction converts one into the other
- **reaction graph** — two reactions joined if they share a metabolite
- and in either case: what do you do about **currency metabolites** (H⁺, H₂O, ATP, NADH…), which
  take part in dozens of reactions and connect everything to everything?

The loader gives you the bipartite graph and both projections, with and without currency metabolites.
Compare them. The network's "properties" depend more on this choice than on the biology.

Source: [BiGG Models](http://bigg.ucsd.edu/models/e_coli_core), Orth et al. 2010.

In [8]:
CURRENCY = {"h", "h2o", "atp", "adp", "amp", "pi", "ppi", "nad", "nadh", "nadp", "nadph", "co2", "o2",
            "coa", "nh4", "q8", "q8h2"}

def load_ecoli_core():
    m = json.loads(get("http://bigg.ucsd.edu/api/v2/models/e_coli_core/download"))
    B = nx.Graph()                                                  # bipartite: metabolite - reaction
    for rxn in m["reactions"]:
        B.add_node(rxn["id"], kind="reaction", name=rxn["name"])
        for met, coef in rxn["metabolites"].items():
            base = met.rsplit("_", 1)[0]                              # strip compartment suffix: atp_c -> atp
            B.add_node(met, kind="metabolite", name=base, currency=base in CURRENCY)
            B.add_edge(met, rxn["id"], role="substrate" if coef < 0 else "product")
    return B

def project(B, onto="metabolite", drop_currency=False):
    '''Collapse the bipartite graph onto one node kind.'''
    keep = [n for n, d in B.nodes(data=True) if d["kind"] == onto and not (drop_currency and d.get("currency"))]
    return nx.bipartite.projected_graph(B, keep) if not drop_currency else \
           nx.bipartite.projected_graph(B.subgraph([n for n, d in B.nodes(data=True)
                                                    if not d.get("currency")]), keep)

B = load_ecoli_core()
print(f"bipartite: {sum(1 for _,d in B.nodes(data=True) if d['kind']=='metabolite')} metabolites, "
      f"{sum(1 for _,d in B.nodes(data=True) if d['kind']=='reaction')} reactions, {B.number_of_edges()} links\n")
describe(project(B, "metabolite"),                     "metabolite graph, currency metabolites kept")
describe(project(B, "metabolite", drop_currency=True), "metabolite graph, currency metabolites removed")
G = project(B, "metabolite", drop_currency=True)

bipartite: 72 metabolites, 95 reactions, 360 links

metabolite graph, currency metabolites kept
  72 nodes, 496 edges, undirected
  mean degree 13.78, max degree 55, average clustering 0.723
  1 connected component(s); largest has 72 nodes (100%)
metabolite graph, currency metabolites removed
  50 nodes, 142 edges, undirected
  mean degree 5.68, max degree 18, average clustering 0.363
  1 connected component(s); largest has 50 nodes (100%)


---
## Exporting to Cytoscape

Whichever network you have as `G`, this writes a SIF file. In Colab it downloads straight to your
computer; elsewhere it lands in the working directory.

In [9]:
to_sif(G, "my_network.sif")
try:
    from google.colab import files; files.download("my_network.sif")
except ImportError:
    pass

wrote my_network.sif
